# Loan Default Prediction — Fixed, Leak-Free, CV-Based Pipeline (373K rows)

Fixes applied vs the original notebook:
1. Train / Validation / Test split — test set is NEVER touched until the final cell.
2. SMOTETomek wrapped INSIDE an imblearn Pipeline + CV — no resampling leakage across folds.
3. SHAP feature importance computed on REAL train data (sampled for speed), not synthetic SMOTE data.
4. Hyperparameter search runs on a 60K-row subsample (SMOTETomek is the slow part at 373K rows) —
   the winning model is refit ONCE on the full training set.
5. Threshold tuned on VALIDATION set only; test set used once, at the end, for honest final metrics.
6. Every step prints progress + timing so you can see exactly where time is going.


In [9]:
import time
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.metrics import (
    classification_report, confusion_matrix, average_precision_score,
    precision_recall_curve, fbeta_score, f1_score
)

from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline  # IMPORTANT: imblearn's Pipeline, not sklearn's

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import shap

RANDOM_STATE = 42

# --- simple timestamped logger so you can see exactly where time goes ---
_start_time = time.time()

def log(msg):
    elapsed = time.time() - _start_time
    print(f"[{elapsed:8.1f}s] {msg}")

log("Imports loaded.")


[     0.0s] Imports loaded.


In [2]:
def clean_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    log(f"Starting cleaning + feature engineering on {df.shape[0]} rows, {df.shape[1]} cols")

    # --- drop identifier columns (must happen first, before anything else) ---
    # UNIQUE_ID is 100% unique per row -- a pure row identifier with zero
    # predictive meaning. Leaving it in lets tree models memorize row
    # identity instead of learning genuine default patterns, which
    # artificially inflates train/val metrics and collapses on true
    # out-of-sample data. Confirmed via nunique() == len(df) check.
    id_columns = ['UNIQUE_ID']
    id_columns_present = [c for c in id_columns if c in df.columns]
    if id_columns_present:
        df.drop(columns=id_columns_present, inplace=True)
        log(f"Step 0/9: Dropped identifier column(s): {id_columns_present}")
    else:
        log("WARNING Step 0/9: expected ID column(s) not found — check id_columns list matches your data.")

    # --- flags ---
    flag_columns = ['SI_FLG', 'LOCKER_HLDR_IND', 'UID_FLG', 'KYC_FLG', 'INB_FLG', 'EKYC_FLG']
    df[flag_columns] = df[flag_columns].replace({'Y': 1, 'N': 0}).astype(float)
    log("Step 1/9: Flag columns converted to 0/1.")

    # --- tenure strings -> months ---
    def convert_to_months(s):
        if pd.isna(s):
            return np.nan
        s = str(s).lower().strip()
        pattern = r'(?:(\d+)\s*yrs?)?\s*(?:(\d+)\s*(?:months|mon))?'
        match = re.match(pattern, s)
        if match:
            years = int(match.group(1)) if match.group(1) else 0
            months = int(match.group(2)) if match.group(2) else 0
            return years * 12 + months
        return np.nan

    df['CREDIT_HISTORY_LENGTH1'] = df['CREDIT_HISTORY_LENGTH1'].apply(convert_to_months)
    df['AVERAGE_ACCT_AGE1'] = df['AVERAGE_ACCT_AGE1'].apply(convert_to_months)
    log("Step 2/9: Tenure strings converted to months.")

    # --- ordinal / categorical maps ---
    income_band_mapping = {
        'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7,
        'H': 8, 'I': 9, 'J': 10, 'K': 11, 'L': 12, 'M': 13, 'EX05': 14
    }
    df['INCOME_BAND1'] = df['INCOME_BAND1'].map(lambda x: income_band_mapping.get(x, np.nan))

    if 'ONEMNTHCR' in df.columns:
        df.rename(columns={'ONEMNTHCR': 'ONEMNTHSCR'}, inplace=True)

    agreg_group_mapping = {
        '#Total Auto Loan': 1, '#Total Xpress Credit': 2,
        '#Housing Loan': 3, '#Education Loan Total': 4
    }
    df['AGREG_GROUP'] = df['AGREG_GROUP'].map(lambda x: agreg_group_mapping.get(x, np.nan))

    product_type_mapping = {'AUTO LOAN': 1, 'PERSONAL LOAN': 2, 'HOME LOAN': 3, 'EDUCATION LOAN': 4}
    df['PRODUCT_TYPE'] = df['PRODUCT_TYPE'].map(lambda x: product_type_mapping.get(x, np.nan))

    month_map = {'JAN': '01', 'FEB': '02', 'MAR': '03', 'APR': '04', 'MAY': '05', 'JUN': '06',
                 'JUL': '07', 'AUG': '08', 'SEP': '09', 'OCT': '10', 'NOV': '11', 'DEC': '12'}

    def convert_time_period(val):
        if pd.isna(val):
            return np.nan
        match = re.match(r'([A-Z]{3})(\d{2})', str(val).upper())
        if match:
            month = month_map.get(match.group(1), '00')
            year = '20' + match.group(2)
            return int(year + month)
        return np.nan

    df['TIME_PERIOD'] = df['TIME_PERIOD'].apply(convert_time_period)
    log("Step 3/9: Ordinal/categorical columns mapped (income band, agreg group, product type, time period).")

    # --- fill known "0 means none" columns ---
    rg_columns = [
        'LAST_1_YR_RG4', 'LAST_3_YR_RG4', 'LAST_1_YR_RG3', 'LAST_1_YR_RG2', 'LAST_1_YR_RG1',
        'FIRST_NPA_TENURE', 'CUST_NO_OF_TIMES_NPA', 'LATEST_NPA_TENURE', 'NO_YRS_NPA',
        'LATEST_RG3_TENURE', 'NO_YRS_RG3', 'TOT_IRAC_CHNG', 'TIMES_IRAC_SLIP', 'TIMES_IRAC_UPR',
        'NO_ENQ', 'CRIFF_11', 'CRIFF_22', 'CRIFF_33', 'CRIFF_44', 'CRIFF_55', 'CRIFF_66', 'TOTAL_CRIFF1'
    ]
    df[rg_columns] = df[rg_columns].fillna(0)
    log("Step 4/9: 'Zero means none' columns filled.")

    # --- spend behaviour features ---
    sdr_cols = [f'{i}MNTHSDR' for i in
                ['ONE', 'TWO', 'THREE', 'FOUR', 'FIVE', 'SIX', 'SEVEN', 'EIGHT', 'NINE', 'TEN', 'ELEVEN', 'TWELVE']]
    df[sdr_cols] = df[sdr_cols].apply(pd.to_numeric, errors='coerce').fillna(0).abs()
    df['ALL_LON_LIMIT'] = pd.to_numeric(df['ALL_LON_LIMIT'], errors='coerce').fillna(0)

    monthly_limit = df['ALL_LON_LIMIT'] / 12
    df_overspend = df[sdr_cols].sub(monthly_limit, axis=0)
    total_spend = df[sdr_cols].sum(axis=1)
    total_overspend = df_overspend.clip(lower=0).sum(axis=1)
    df['overspend_ratio'] = total_overspend / (total_spend + 1e-6)

    limit_matrix = pd.DataFrame(
        np.tile(monthly_limit.values[:, None], (1, len(sdr_cols))),
        columns=sdr_cols, index=df.index
    )
    overspend_flags = df[sdr_cols] > limit_matrix

    def max_consecutive_true(arr):
        max_streak = streak = 0
        for val in arr:
            if val:
                streak += 1
                max_streak = max(max_streak, streak)
            else:
                streak = 0
        return max_streak

    df['max_consec_overspend'] = overspend_flags.apply(max_consecutive_true, axis=1)
    log("Step 5/9: Overspend ratio + max consecutive overspend streak computed.")

    # --- outstanding balance trend ---
    out_cols = [f'{i}MNTHOUTSTANGBAL' for i in
                ['TWELVE', 'ELEVEN', 'TEN', 'NINE', 'EIGHT', 'SEVEN', 'SIX', 'FIVE', 'FOUR', 'THREE', 'TWO', 'ONE']]
    df[out_cols] = df[out_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    def calc_slope(row):
        x = np.arange(12)
        y = row.values
        return np.polyfit(x, y, 1)[0]

    df['outbal_slope'] = df[out_cols].apply(calc_slope, axis=1)
    df['outbal_is_declining'] = (df['outbal_slope'] < 0).astype(int)
    log("Step 6/9: Outstanding balance slope computed.")

    # --- MTD debit trend ---
    term_debit_cols = [f"{i}MNTHAVGMTD" for i in
                        ['ONE', 'TWO', 'THREE', 'FOUR', 'FIVE', 'SIX', 'SEVEN', 'EIGHT', 'NINE', 'TEN', 'ELEVEN', 'TWELVE']]
    df[term_debit_cols] = df[term_debit_cols].fillna(0)
    df['slope_MTD'] = df[term_debit_cols].apply(calc_slope, axis=1)
    df['is_debit_declining_MTD'] = (df['slope_MTD'] < 0).astype(int)
    log("Step 7/9: MTD debit slope computed.")

    # --- drop raw columns replaced by engineered features ---
    keywords_to_remove = ['SDR', 'SCR', 'OUTSTANGBAL', 'AVGMTD', 'AVGQTD', 'AVGYTD']
    exceptions = ['KYC_SCR']
    cols_to_drop = [c for c in df.columns if any(kw in c for kw in keywords_to_remove) and c not in exceptions]
    df.drop(columns=cols_to_drop, inplace=True)
    log(f"Step 8/9: Dropped {len(cols_to_drop)} raw columns replaced by engineered features.")

    log(f"Step 9/9: Cleaning + feature engineering DONE. Final shape: {df.shape}")
    return df


In [3]:
log("Loading raw CSV...")
df = pd.read_csv("HACKATHON_TRAINING_DATA.CSV")
log(f"Raw data loaded: {df.shape[0]} rows, {df.shape[1]} cols")

df = clean_and_engineer(df)

df.to_csv("cleaned_data.csv", index=False)
log("Cleaned data saved to cleaned_data.csv")


[     3.7s] Loading raw CSV...
[    17.6s] Raw data loaded: 327741 rows, 139 cols
[    18.4s] Starting cleaning + feature engineering on 327741 rows, 139 cols
[    18.4s] Step 0/9: Dropped identifier column(s): ['UNIQUE_ID']
[    19.6s] Step 1/9: Flag columns converted to 0/1.
[    23.3s] Step 2/9: Tenure strings converted to months.
[    25.9s] Step 3/9: Ordinal/categorical columns mapped (income band, agreg group, product type, time period).
[    26.0s] Step 4/9: 'Zero means none' columns filled.
[    31.0s] Step 5/9: Overspend ratio + max consecutive overspend streak computed.
[    76.2s] Step 6/9: Outstanding balance slope computed.
[   121.3s] Step 7/9: MTD debit slope computed.
[   121.3s] Step 8/9: Dropped 72 raw columns replaced by engineered features.
[   121.3s] Step 9/9: Cleaning + feature engineering DONE. Final shape: (327741, 72)
[   157.4s] Cleaned data saved to cleaned_data.csv


In [4]:
X = df.drop(columns=['TARGET'])
y = df['TARGET']

log("Splitting into train (60%) / val (20%) / test (20%), stratified...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

log(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
log(f"Train default rate: {y_train.mean():.4f} | Val: {y_val.mean():.4f} | Test: {y_test.mean():.4f}")


[   206.7s] Splitting into train (60%) / val (20%) / test (20%), stratified...
[   208.0s] Train: (196644, 71), Val: (65548, 71), Test: (65549, 71)
[   208.0s] Train default rate: 0.1081 | Val: 0.1081 | Test: 0.1081


In [5]:
log("Fitting SimpleImputer (median) on train, transforming train/val/test...")
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_imp = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)
log("Imputation done.")

log("Fitting Yeo-Johnson power transform on train, transforming train/val/test...")
numeric_cols = X_train_imp.select_dtypes(include=[np.number]).columns
yeo = PowerTransformer(method='yeo-johnson', standardize=False)
X_train_imp[numeric_cols] = yeo.fit_transform(X_train_imp[numeric_cols])
X_val_imp[numeric_cols] = yeo.transform(X_val_imp[numeric_cols])
X_test_imp[numeric_cols] = yeo.transform(X_test_imp[numeric_cols])
log("Yeo-Johnson transform done.")

log("Fitting MinMaxScaler on train, transforming train/val/test...")
scaler = MinMaxScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train.columns, index=X_train.index)
X_val_s = pd.DataFrame(scaler.transform(X_val_imp), columns=X_val.columns, index=X_val.index)
X_test_s = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test.columns, index=X_test.index)
log("Scaling done. Preprocessing complete.")


[   216.6s] Fitting SimpleImputer (median) on train, transforming train/val/test...
[   220.7s] Imputation done.
[   220.7s] Fitting Yeo-Johnson power transform on train, transforming train/val/test...
[   406.0s] Yeo-Johnson transform done.
[   406.0s] Fitting MinMaxScaler on train, transforming train/val/test...
[   406.6s] Scaling done. Preprocessing complete.


In [6]:
log("Fitting XGBoost feature-selection model on real train data (scale_pos_weight for imbalance)...")
fs_model = XGBClassifier(
    eval_metric='logloss',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
fs_model.fit(X_train_s, y_train)
log("Feature-selection model fit done.")

SHAP_SAMPLE_SIZE = 5000
log(f"Sampling {SHAP_SAMPLE_SIZE} stratified rows for SHAP (full {len(X_train_s)}-row SHAP would be very slow)...")
shap_sample_idx = (
    pd.Series(y_train.values, index=X_train_s.index)
    .groupby(y_train.values)
    .apply(lambda s: s.sample(n=min(len(s), SHAP_SAMPLE_SIZE // 2), random_state=RANDOM_STATE).index)
)
shap_sample_idx = pd.Index(np.concatenate(shap_sample_idx.values))
X_train_shap_sample = X_train_s.loc[shap_sample_idx]
log(f"SHAP sample ready: {X_train_shap_sample.shape[0]} rows.")

log("Computing SHAP values (this is usually the slowest single step)...")
explainer = shap.Explainer(fs_model)
shap_values = explainer(X_train_shap_sample)
shap_importance = np.abs(shap_values.values).mean(axis=0)
log("SHAP values computed.")

shap_feature_importance = pd.DataFrame({
    'feature': X_train_shap_sample.columns,
    'importance': shap_importance
}).sort_values('importance', ascending=False)

TOP_N = 30
top_features = shap_feature_importance['feature'].head(TOP_N).tolist()
log(f"Top {TOP_N} features selected.")
print(shap_feature_importance.head(TOP_N).to_string(index=False))

X_train_top = X_train_s[top_features]
X_val_top = X_val_s[top_features]
X_test_top = X_test_s[top_features]
log("Feature selection complete. Train/val/test reduced to top features.")


[   412.1s] Fitting XGBoost feature-selection model on real train data (scale_pos_weight for imbalance)...
[   417.6s] Feature-selection model fit done.
[   417.6s] Sampling 5000 stratified rows for SHAP (full 196644-row SHAP would be very slow)...
[   417.7s] SHAP sample ready: 5000 rows.
[   417.7s] Computing SHAP values (this is usually the slowest single step)...
[   419.4s] SHAP values computed.
[   419.4s] Top 30 features selected.
               feature  importance
              CRIFF_11    1.333480
              CRIFF_22    0.259235
        LATEST_CR_DAYS    0.248931
     LATEST_RG3_TENURE    0.216924
         LAST_1_YR_RG2    0.210668
       TIMES_IRAC_SLIP    0.204049
          ALL_LON_OUTS    0.167432
    PRI_OVERDUE_ACCTS1    0.166176
              CRIFF_55    0.148158
OLDEST_RESIDUAL_TENURE    0.130871
              CRIFF_33    0.128876
         ALL_LON_LIMIT    0.127150
        LATEST_DR_DAYS    0.124022
          outbal_slope    0.113215
              CRIFF_66    0.11251

In [10]:
SEARCH_SAMPLE_SIZE = 60_000  # rows used for hyperparameter search only

if len(X_train_top) > SEARCH_SAMPLE_SIZE:
    log(f"Subsampling {SEARCH_SAMPLE_SIZE} rows (of {len(X_train_top)}) for hyperparameter search...")
    X_search, _, y_search, _ = train_test_split(
        X_train_top, y_train,
        train_size=SEARCH_SAMPLE_SIZE,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )
else:
    X_search, y_search = X_train_top, y_train

log(f"Search subsample ready: {len(X_search)} rows.")

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

candidates = {
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMClassifier(objective='binary', random_state=RANDOM_STATE, verbosity=-1, n_jobs=-1),
    "CatBoost": CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, thread_count=-1),
}

param_grids = {
    "XGBoost": {
        "clf__n_estimators": [200, 400],
        "clf__max_depth": [4, 6, 8],
        "clf__learning_rate": [0.03, 0.1],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
    },
    "LightGBM": {
        "clf__n_estimators": [200, 400],
        "clf__num_leaves": [15, 31, 63],
        "clf__learning_rate": [0.03, 0.1],
        "clf__reg_alpha": [0.0, 1.0],
        "clf__reg_lambda": [0.0, 1.0],
    },
    "CatBoost": {
        "clf__iterations": [200, 400],
        "clf__depth": [4, 6, 8],
        "clf__learning_rate": [0.03, 0.1],
    },
}

results = {}
best_params = {}

for name, model in candidates.items():
    log(f"Starting hyperparameter search for {name}...")
    pipe = ImbPipeline(steps=[
        ("smote_tomek", SMOTETomek(random_state=RANDOM_STATE, n_jobs=-1)),
        ("clf", model),
    ])

    # RandomizedSearchCV — NOTE: HalvingRandomSearchCV was tried here first
    # but is a bad fit for imbalanced data + SMOTE. Halving starts with a
    # small row subset and grows it for promising candidates only; on
    # imbalanced data that small starting subset can contain just 1 (or 0)
    # minority rows, and SMOTE cannot run without enough minority samples
    # to find neighbors from — causing "All fits failed" errors. Plain
    # RandomizedSearchCV always uses the full X_search subsample, so every
    # fold has enough minority rows for SMOTE to work correctly.
    search = RandomizedSearchCV(
        pipe,
        param_distributions=param_grids[name],
        n_iter=6,
        scoring="average_precision",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=False,
    )
    search.fit(X_search, y_search)

    best_params[name] = search.best_params_
    results[name] = search.best_score_
    log(f"{name} search DONE — best CV PR-AUC (subsample) = {search.best_score_:.4f} | params = {search.best_params_}")

best_model_name = max(results, key=results.get)
log(f"Best model by CV PR-AUC: {best_model_name} ({results[best_model_name]:.4f})")

[    12.4s] Subsampling 60000 rows (of 196644) for hyperparameter search...
[    12.7s] Search subsample ready: 60000 rows.
[    12.7s] Starting hyperparameter search for XGBoost...
[   297.4s] XGBoost search DONE — best CV PR-AUC (subsample) = 0.5831 | params = {'clf__subsample': 1.0, 'clf__n_estimators': 400, 'clf__max_depth': 6, 'clf__learning_rate': 0.1, 'clf__colsample_bytree': 1.0}
[   297.4s] Starting hyperparameter search for LightGBM...
[   569.2s] LightGBM search DONE — best CV PR-AUC (subsample) = 0.5889 | params = {'clf__reg_lambda': 1.0, 'clf__reg_alpha': 1.0, 'clf__num_leaves': 31, 'clf__n_estimators': 400, 'clf__learning_rate': 0.1}
[   569.2s] Starting hyperparameter search for CatBoost...
[   842.8s] CatBoost search DONE — best CV PR-AUC (subsample) = 0.5531 | params = {'clf__learning_rate': 0.1, 'clf__iterations': 200, 'clf__depth': 8}
[   842.8s] Best model by CV PR-AUC: LightGBM (0.5889)


In [11]:
log(f"Refitting {best_model_name} on the FULL {len(X_train_top)}-row train set with its best params (runs once)...")

final_pipe = ImbPipeline(steps=[
    ("smote_tomek", SMOTETomek(random_state=RANDOM_STATE, n_jobs=-1)),
    ("clf", candidates[best_model_name]),
])
final_pipe.set_params(**best_params[best_model_name])
final_pipe.fit(X_train_top, y_train)
best_model = final_pipe

log(f"Full-data refit of {best_model_name} complete.")


[   881.6s] Refitting LightGBM on the FULL 196644-row train set with its best params (runs once)...
[  1139.5s] Full-data refit of LightGBM complete.


In [12]:
log("Scoring validation set...")
val_probs = best_model.predict_proba(X_val_top)[:, 1]

log("Sweeping thresholds on validation set to find best F1...")
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores[:-1])]

log(f"Best threshold (chosen on validation set): {best_thresh:.4f}")
log(f"Validation F1 at this threshold: {max(f1_scores):.4f}")


[  1179.3s] Scoring validation set...
[  1179.6s] Sweeping thresholds on validation set to find best F1...
[  1179.6s] Best threshold (chosen on validation set): 0.2874
[  1179.6s] Validation F1 at this threshold: 0.5952


In [13]:
log("Scoring TEST set (touched for the first and only time here)...")
test_probs = best_model.predict_proba(X_test_top)[:, 1]
y_pred = (test_probs >= best_thresh).astype(int)

log("Final test evaluation complete.")

print("\n===== FINAL TEST SET RESULTS (never seen before this point) =====")
print(f"Model: {best_model_name}")
print(f"PR-AUC (test): {average_precision_score(y_test, test_probs):.4f}")
print(f"F1  (test): {f1_score(y_test, y_pred):.4f}")
print(f"F2  (test): {fbeta_score(y_test, y_pred, beta=2):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

log("ALL DONE.")


[  1190.0s] Scoring TEST set (touched for the first and only time here)...
[  1190.3s] Final test evaluation complete.

===== FINAL TEST SET RESULTS (never seen before this point) =====
Model: LightGBM
PR-AUC (test): 0.6397
F1  (test): 0.5939
F2  (test): 0.6311

Confusion Matrix:
 [[54496  3965]
 [ 2420  4668]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.93      0.94     58461
           1       0.54      0.66      0.59      7088

    accuracy                           0.90     65549
   macro avg       0.75      0.80      0.77     65549
weighted avg       0.91      0.90      0.91     65549

[  1190.4s] ALL DONE.


In [14]:
from sklearn.metrics import roc_auc_score
print(f"ROC-AUC (test): {roc_auc_score(y_test, test_probs):.4f}")

ROC-AUC (test): 0.9164
